In [1]:
!pip install -q datasets transformers tokenizers torch wandb scikit-learn

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

import numpy as np
import math
import random
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

🚀 Using device: cuda
   GPU: Tesla T4
   Memory: 15.83 GB


In [25]:
class Config:
    """Hyperparameters and configuration"""
    # Model architecture
    vocab_size = 30522  # BERT tokenizer vocab size
    hidden_size = 256
    num_layers = 4
    num_heads = 8
    intermediate_size = 1024
    max_position_embeddings = 512
    dropout = 0.1
    pooling_strategy = 'mean'  # Options: 'mean', 'max', 'cls'

    # Training
    batch_size = 32
    num_epochs = 3
    learning_rate = 5e-5
    warmup_steps = 500
    max_grad_norm = 1.0
    temperature = 0.05  # For contrastive loss

    # Data
    max_length = 128
    dataset_name = "JYumeko/processed_pubmed_scientific_papers"
    dataset_config = "arxiv"
    num_samples = 50000  # Use subset for faster training

    # Evaluation
    num_eval_queries = 5
    top_k = 5

    # Logging
    use_wandb = False  # Set to True to enable W&B logging
    log_steps = 100
    eval_steps = 1000

config = Config()

In [27]:
def load_scientific_dataset(config):
    """
    Load and preprocess scientific papers dataset
    """
    print(f"📚 Loading dataset: {config.dataset_name}/{config.dataset_config}")

    # Load dataset from Hugging Face
    # The traceback indicates that 'arxiv' is not a valid config for this dataset.
    # Available: ['default']
    # Also, trust_remote_code is deprecated.
    dataset = load_dataset(
        config.dataset_name,
        # Use the 'default' config as indicated by the error message
        'default',
        split='train',
        # Remove deprecated argument
        # trust_remote_code=True
    )

    # Use subset for faster experimentation
    if config.num_samples and config.num_samples < len(dataset):
        dataset = dataset.select(range(config.num_samples))

    print(f"   ✓ Loaded {len(dataset)} samples")
    print(f"   Sample keys: {dataset.column_names}")

    return dataset

In [6]:
def create_tokenizer():
    """
    Initialize tokenizer (using pretrained BERT tokenizer for vocabulary)
    """
    print("🔤 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    print(f"   ✓ Vocab size: {len(tokenizer)}")
    return tokenizer

In [7]:
class ContrastiveDataset(Dataset):
    """
    Dataset for contrastive learning (SimCSE-style)
    Creates two views of each text through different dropout masks
    """
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize (will create two views via dropout during forward pass)
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }


In [8]:
def prepare_data(config):
    """
    Complete data preparation pipeline
    """
    # Load dataset
    dataset = load_scientific_dataset(config)

    # Extract abstracts (or full text if available)
    texts = []
    for item in tqdm(dataset, desc="Extracting texts"):
        # Try different field names
        if 'abstract' in item and item['abstract']:
            texts.append(item['abstract'])
        elif 'summary' in item and item['summary']:
            # Use first 500 chars of article if no abstract
            texts.append(item['summary'][:500])

    print(f"   ✓ Extracted {len(texts)} text samples")

    # Create tokenizer
    tokenizer = create_tokenizer()

    # Create dataset
    train_size = int(0.9 * len(texts))
    train_texts = texts[:train_size]
    val_texts = texts[train_size:]

    train_dataset = ContrastiveDataset(train_texts, tokenizer, config.max_length)
    val_dataset = ContrastiveDataset(val_texts, tokenizer, config.max_length)

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"   ✓ Train batches: {len(train_loader)}")
    print(f"   ✓ Val batches: {len(val_loader)}")

    return train_loader, val_loader, tokenizer, val_texts



In [9]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [29]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism"""
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        assert hidden_size % num_heads == 0

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_linear = nn.Linear(hidden_size, hidden_size)
        self.k_linear = nn.Linear(hidden_size, hidden_size)
        self.v_linear = nn.Linear(hidden_size, hidden_size)
        self.out_linear = nn.Linear(hidden_size, hidden_size)

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # Linear projections and reshape to (batch, num_heads, seq_len, head_dim)
        q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale

        # Apply mask
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
            # Use a smaller negative value for masking with mixed precision
            scores = scores.masked_fill(mask == 0, -1e4)

        # Attention weights
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values
        context = torch.matmul(attn_weights, v)

        # Reshape and project
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        output = self.out_linear(context)

        return output

In [11]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network"""
    def __init__(self, hidden_size, intermediate_size, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, intermediate_size)
        self.fc2 = nn.Linear(intermediate_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [12]:
class TransformerEncoderLayer(nn.Module):
    """Single transformer encoder layer"""
    def __init__(self, hidden_size, num_heads, intermediate_size, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(hidden_size, num_heads, dropout)
        self.feed_forward = FeedForward(hidden_size, intermediate_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x


In [13]:
class EmbeddingModel(nn.Module):
    """
    Complete Transformer Encoder model for generating embeddings
    """
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embeddings
        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_encoding = PositionalEncoding(config.hidden_size, config.max_position_embeddings)

        # Transformer encoder layers
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                config.hidden_size,
                config.num_heads,
                config.intermediate_size,
                config.dropout
            )
            for _ in range(config.num_layers)
        ])

        self.dropout = nn.Dropout(config.dropout)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=0.02)

    def pool_embeddings(self, hidden_states, attention_mask):
        """Apply pooling strategy to get fixed-size embeddings"""
        if self.config.pooling_strategy == 'cls':
            # Use first token (CLS token)
            return hidden_states[:, 0]

        elif self.config.pooling_strategy == 'mean':
            # Mean pooling with attention mask
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            return sum_embeddings / sum_mask

        elif self.config.pooling_strategy == 'max':
            # Max pooling
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            hidden_states[mask_expanded == 0] = -1e9
            return torch.max(hidden_states, dim=1)[0]

    def forward(self, input_ids, attention_mask=None):
        """
        Forward pass
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        Returns:
            embeddings: (batch_size, hidden_size)
        """
        # Token embeddings + positional encoding
        x = self.token_embedding(input_ids)
        x = self.position_encoding(x)
        x = self.dropout(x)

        # Pass through transformer layers
        for layer in self.layers:
            x = layer(x, attention_mask)

        # Pool to get fixed-size embeddings
        embeddings = self.pool_embeddings(x, attention_mask)

        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)

        return embeddings


In [14]:
def build_model(config):
    """Initialize model"""
    print("🏗️  Building model...")
    model = EmbeddingModel(config)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"   ✓ Total parameters: {total_params:,}")
    print(f"   ✓ Trainable parameters: {trainable_params:,}")
    print(f"   ✓ Model size: ~{total_params * 4 / 1e6:.2f} MB")

    return model.to(device)

In [15]:
class ContrastiveLoss(nn.Module):
    """
    SimCSE-style contrastive loss using in-batch negatives
    """
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings1, embeddings2):
        """
        Args:
            embeddings1: (batch_size, hidden_size) - first view
            embeddings2: (batch_size, hidden_size) - second view
        Returns:
            loss: scalar
        """
        batch_size = embeddings1.size(0)

        # Compute similarity matrix: (batch_size, batch_size)
        # Each row i contains similarities between embeddings1[i] and all embeddings2
        similarity_matrix = torch.matmul(embeddings1, embeddings2.T) / self.temperature

        # Labels: positive pairs are on the diagonal
        labels = torch.arange(batch_size, device=embeddings1.device)

        # Cross-entropy loss (treats other samples as negatives)
        loss = F.cross_entropy(similarity_matrix, labels)

        return loss

In [16]:
def train_model(model, train_loader, val_loader, config):
    """
    Main training loop with contrastive learning
    """
    print("\n" + "="*70)
    print("🚂 Starting Training")
    print("="*70)

    # Initialize optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config.learning_rate)
    total_steps = len(train_loader) * config.num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=total_steps
    )

    # Loss function
    criterion = ContrastiveLoss(temperature=config.temperature)

    # Mixed precision training
    scaler = GradScaler()

    # Training metrics
    global_step = 0
    best_val_loss = float('inf')

    # Optional W&B logging
    if config.use_wandb:
        import wandb
        wandb.init(project="scientific-embeddings", config=vars(config))
        wandb.watch(model)

    for epoch in range(config.num_epochs):
        print(f"\n📅 Epoch {epoch + 1}/{config.num_epochs}")
        model.train()
        epoch_loss = 0

        progress_bar = tqdm(train_loader, desc=f"Training")

        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass twice with different dropout masks (SimCSE approach)
            with autocast():
                embeddings1 = model(input_ids, attention_mask)
                embeddings2 = model(input_ids, attention_mask)  # Different dropout

                loss = criterion(embeddings1, embeddings2)

            # Backward pass with mixed precision
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            # Update metrics
            epoch_loss += loss.item()
            global_step += 1

            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })

            # Logging
            if global_step % config.log_steps == 0:
                avg_loss = epoch_loss / (batch_idx + 1)
                print(f"   Step {global_step}: Loss = {avg_loss:.4f}")

                if config.use_wandb:
                    wandb.log({
                        'train/loss': loss.item(),
                        'train/learning_rate': scheduler.get_last_lr()[0],
                        'train/epoch': epoch
                    }, step=global_step)

            # Validation
            if global_step % config.eval_steps == 0:
                val_loss = evaluate(model, val_loader, criterion, config)
                print(f"   📊 Validation Loss: {val_loss:.4f}")

                if config.use_wandb:
                    wandb.log({'val/loss': val_loss}, step=global_step)

                # Save best model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), 'best_model.pt')
                    print("   ✓ Saved best model!")

                model.train()

        # Epoch summary
        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"\n   Epoch {epoch + 1} Summary:")
        print(f"   - Average Loss: {avg_epoch_loss:.4f}")
        print(f"   - Best Val Loss: {best_val_loss:.4f}")

    print("\n" + "="*70)
    print("✅ Training Complete!")
    print("="*70)

    return model


In [17]:
def evaluate(model, val_loader, criterion, config):
    """Evaluate model on validation set"""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            embeddings1 = model(input_ids, attention_mask)
            embeddings2 = model(input_ids, attention_mask)

            loss = criterion(embeddings1, embeddings2)
            total_loss += loss.item()

    return total_loss / len(val_loader)

In [18]:
def evaluate_embeddings(model, tokenizer, texts, config):
    """
    Evaluate embeddings through similarity search
    """
    print("\n" + "="*70)
    print("🔍 Evaluating Embeddings - Similarity Search")
    print("="*70)

    model.eval()

    # Select random queries
    query_indices = random.sample(range(len(texts)), config.num_eval_queries)

    # Encode all texts
    print("\n📊 Encoding all documents...")
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), config.batch_size)):
            batch_texts = texts[i:i + config.batch_size]

            # Tokenize
            encoding = tokenizer(
                batch_texts,
                max_length=config.max_length,
                padding=True,
                truncation=True,
                return_tensors='pt'
            )

            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)

            # Get embeddings
            embeddings = model(input_ids, attention_mask)
            all_embeddings.append(embeddings.cpu().numpy())

    all_embeddings = np.vstack(all_embeddings)
    print(f"   ✓ Encoded {len(all_embeddings)} documents")
    print(f"   ✓ Embedding shape: {all_embeddings.shape}")

    # Perform similarity search for each query
    print(f"\n🔎 Running {config.num_eval_queries} similarity searches...")
    print("="*70)

    for query_idx in query_indices:
        query_text = texts[query_idx]
        query_embedding = all_embeddings[query_idx:query_idx+1]

        # Compute similarities
        similarities = cosine_similarity(query_embedding, all_embeddings)[0]

        # Get top-k (excluding the query itself)
        top_indices = np.argsort(similarities)[::-1][1:config.top_k+1]

        # Display results
        print(f"\n📄 Query {query_idx}:")
        print(f"   {query_text[:200]}...")
        print(f"\n   Top-{config.top_k} Similar Documents:")

        for rank, idx in enumerate(top_indices, 1):
            similarity_score = similarities[idx]
            retrieved_text = texts[idx]
            print(f"\n   {rank}. [Similarity: {similarity_score:.4f}]")
            print(f"      {retrieved_text[:150]}...")

        print("\n" + "-"*70)

    return all_embeddings



In [19]:
def save_model(model, tokenizer, config, path='./embedding_model'):
    """Save trained model and tokenizer"""
    import os
    os.makedirs(path, exist_ok=True)

    # Save model
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': vars(config)
    }, f'{path}/model.pt')

    # Save tokenizer
    tokenizer.save_pretrained(path)

    print(f"✅ Model saved to {path}")


def load_model(path='./embedding_model'):
    """Load trained model"""
    checkpoint = torch.load(f'{path}/model.pt')
    config_dict = checkpoint['config']

    # Reconstruct config
    config = Config()
    for key, value in config_dict.items():
        setattr(config, key, value)

    # Load model
    model = EmbeddingModel(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(path)

    print(f"✅ Model loaded from {path}")
    return model, tokenizer, config

In [20]:
def main():
    """Main execution pipeline"""
    print("\n" + "="*70)
    print("🔬 Scientific Paper Embedding Model Training")
    print("="*70)

    # 1. Prepare data
    print("\n📦 Step 1: Data Preparation")
    train_loader, val_loader, tokenizer, val_texts = prepare_data(config)

    # 2. Build model
    print("\n🏗️  Step 2: Model Construction")
    model = build_model(config)

    # 3. Train model
    print("\n🚂 Step 3: Model Training")
    trained_model = train_model(model, train_loader, val_loader, config)

    # 4. Evaluate embeddings
    print("\n📊 Step 4: Embedding Evaluation")
    embeddings = evaluate_embeddings(trained_model, tokenizer, val_texts, config)

    # 5. Save model
    print("\n💾 Step 5: Saving Model")
    save_model(trained_model, tokenizer, config)

    print("\n" + "="*70)
    print("🎉 Pipeline Complete!")
    print("="*70)
    print("\n📝 Next Steps:")
    print("   - Fine-tune hyperparameters in Config class")
    print("   - Experiment with different pooling strategies")
    print("   - Try larger datasets or different scientific domains")
    print("   - Use trained embeddings for downstream tasks")
    print("\n💡 To use the model:")
    print("   model, tokenizer, config = load_model('./embedding_model')")

    return trained_model, tokenizer, embeddings


In [30]:
if __name__ == "__main__":
    trained_model, tokenizer, embeddings = main()


🔬 Scientific Paper Embedding Model Training

📦 Step 1: Data Preparation
📚 Loading dataset: JYumeko/processed_pubmed_scientific_papers/arxiv
   ✓ Loaded 50000 samples
   Sample keys: ['abstract', 'summary']


Extracting texts:   0%|          | 0/50000 [00:00<?, ?it/s]

   ✓ Extracted 48816 text samples
🔤 Loading tokenizer...
   ✓ Vocab size: 30522
   ✓ Train batches: 1373
   ✓ Val batches: 153

🏗️  Step 2: Model Construction
🏗️  Building model...
   ✓ Total parameters: 10,972,672
   ✓ Trainable parameters: 10,972,672
   ✓ Model size: ~43.89 MB

🚂 Step 3: Model Training

🚂 Starting Training

📅 Epoch 1/3


Training:   0%|          | 0/1373 [00:00<?, ?it/s]

   Step 100: Loss = 3.4443
   Step 200: Loss = 3.4377
   Step 300: Loss = 3.4275
   Step 400: Loss = 3.4294
   Step 500: Loss = 3.4300
   Step 600: Loss = 3.4305
   Step 700: Loss = 3.4297
   Step 800: Loss = 3.4266
   Step 900: Loss = 3.4280
   Step 1000: Loss = 3.4248
   📊 Validation Loss: 3.2697
   ✓ Saved best model!
   Step 1100: Loss = 3.4018
   Step 1200: Loss = 3.3504
   Step 1300: Loss = 3.2823

   Epoch 1 Summary:
   - Average Loss: 3.2338
   - Best Val Loss: 3.2697

📅 Epoch 2/3


...

   Step 1400: Loss = 2.3531
   Step 1500: Loss = 2.2305
   Step 1600: Loss = 2.2140
   Step 1700: Loss = 2.1928
   Step 1800: Loss = 2.1658
   Step 1900: Loss = 2.1438
   Step 2000: Loss = 2.1288
   📊 Validation Loss: 2.0623
   ✓ Saved best model!
   Step 2100: Loss = 2.1112
   Step 2200: Loss = 2.0929
   Step 2300: Loss = 2.0757
   Step 2400: Loss = 2.0589
   Step 2500: Loss = 2.0359
   Step 2600: Loss = 2.0072
   Step 2700: Loss = 1.9774

   Epoch 2 Summary:
   - Average Loss: 1.9643
   - Best Val Loss: 2.0623

📅 Epoch 3/3


Training:   0%|          | 0/1373 [00:00<?, ?it/s]

   Step 2800: Loss = 1.5872
   Step 2900: Loss = 1.5826
   Step 3000: Loss = 1.5749
   📊 Validation Loss: 1.4206
   ✓ Saved best model!
   Step 3100: Loss = 1.5700
   Step 3200: Loss = 1.5679
   Step 3300: Loss = 1.5633
   Step 3400: Loss = 1.5568
   Step 3500: Loss = 1.5526
   Step 3600: Loss = 1.5475
   Step 3700: Loss = 1.5435
   Step 3800: Loss = 1.5406
   Step 3900: Loss = 1.5375
   Step 4000: Loss = 1.5344
   📊 Validation Loss: 1.3657
   ✓ Saved best model!
   Step 4100: Loss = 1.5315

   Epoch 3 Summary:
   - Average Loss: 1.5312
   - Best Val Loss: 1.3657

✅ Training Complete!

📊 Step 4: Embedding Evaluation

🔍 Evaluating Embeddings - Similarity Search

📊 Encoding all documents...


  0%|          | 0/153 [00:00<?, ?it/s]

   ✓ Encoded 4882 documents
   ✓ Embedding shape: (4882, 256)

🔎 Running 5 similarity searches...

📄 Query 912:
   nasopharyngeal angiofibroma na rare vascular tumor represents 0.05 head neck tumors time although histologically benign shows locally aggressive growth bone destruction spread natural foramina fissure...

   Top-5 Similar Documents:

   1. [Similarity: 0.9999]
      osteomas slow growing innocuous benign osteogenic tumours composed compact and/or cancellous bone the central osteoma arises endosteum peripheral oste...

   2. [Similarity: 0.9999]
      recently improvement endoscopic skills detection gastrointestinal gi submucosal tumors smts gastrointestinal stromal tumors gists increasing worldwide...

   3. [Similarity: 0.9999]
      systemic metastases occur advanced cases commonest sites lung liver bone presence intracranial mass often life threatening warrants prompt treatment r...

   4. [Similarity: 0.9999]
      synovial sarcoma ss mesenchymal spindle cell tumor var